# ESG & Financial Performance Analysis

This notebook explores the relationship between ESG scores (Environmental, Social, Governance) and key financial metrics (Profit Margin, Revenue, Market Cap, Growth Rate) using the Kaggle dataset "Synthetic ESG & Financial Performance Dataset" by Shreyash Jagtap.

**Target audience:** Investors and corporate sustainability analysts who want to assess ESG as a non-financial risk factor.

**Analysis workflow:**
1. Load and inspect data
2. Clean and preprocess
3. Create ESG categories (Low/Medium/High)
4. Visualize relationships
5. Draw conclusions and note limitations

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better visuals
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load and Inspect Data

In [ ]:
# Load dataset
df = pd.read_csv('esg_data.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
# Statistical summary of numeric columns
numeric_cols = ['ESG_Overall', 'ESG_Environmental', 'ESG_Social', 'ESG_Governance', 
                'ProfitMargin', 'Revenue', 'MarketCap', 'GrowthRate']
df[numeric_cols].describe()

## 3. Data Cleaning & Preprocessing

We drop rows where key numeric columns are missing and create an `ESG_Category` column based on tertiles of `ESG_Overall`.

In [ ]:
# Drop rows with missing values in key columns
df_clean = df.dropna(subset=['ESG_Overall', 'ProfitMargin', 'Revenue', 'MarketCap'])
print(f"Rows after dropping missing: {len(df_clean)}")

# Create ESG category based on tertiles
df_clean['ESG_Category'] = pd.qcut(df_clean['ESG_Overall'], q=3, labels=['Low', 'Medium', 'High'])
df_clean['ESG_Category'].value_counts()

## 4. Visualization & Analysis

We replicate the six charts from the Streamlit dashboard using static plots.

### 4.1 ESG Overall Score vs Profit Margin

Scatter plot colored by industry, with overlaid trend line.

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
sns.scatterplot(data=df_clean, x='ESG_Overall', y='ProfitMargin', hue='Industry', alpha=0.7, ax=ax)

# Add linear trend line
z = np.polyfit(df_clean['ESG_Overall'], df_clean['ProfitMargin'], 1)
p = np.poly1d(z)
x_trend = np.sort(df_clean['ESG_Overall'])
ax.plot(x_trend, p(x_trend), 'k--', linewidth=1, label=f'Trend (slope={z[0]:.3f})')

ax.set_xlabel('ESG Overall Score')
ax.set_ylabel('Profit Margin (%)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 4.2 Average Profit Margin by ESG Category (with Standard Deviation Error Bars)

In [ ]:
# Group and calculate mean and std
avg_profit = df_clean.groupby('ESG_Category')['ProfitMargin'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(8,5))
sns.barplot(data=avg_profit, x='ESG_Category', y='mean', palette='viridis', ax=ax)
ax.errorbar(x=range(len(avg_profit)), y=avg_profit['mean'], yerr=avg_profit['std'],
            fmt='none', c='black', capsize=5)
for i, row in avg_profit.iterrows():
    ax.text(i, row['mean'] + 0.5, f"{row['mean']:.1f}%", ha='center')
ax.set_ylabel('Average Profit Margin (%)')
ax.set_title('Profit Margin by ESG Category')
plt.tight_layout()
plt.show()

### 4.3 Profit Margin Trend Over Time by ESG Category

In [ ]:
trend_data = df_clean.groupby(['Year', 'ESG_Category'])['ProfitMargin'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10,5))
sns.lineplot(data=trend_data, x='Year', y='ProfitMargin', hue='ESG_Category', marker='o', ax=ax)
ax.set_ylabel('Average Profit Margin (%)')
ax.set_title('Profit Margin Trend by ESG Category')
plt.tight_layout()
plt.show()

### 4.4 ESG Overall Score Trend by Industry

In [ ]:
trend_esg = df_clean.groupby(['Year', 'Industry'])['ESG_Overall'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12,5))
sns.lineplot(data=trend_esg, x='Year', y='ESG_Overall', hue='Industry', marker='o', ax=ax)
ax.set_ylabel('Average ESG Overall Score')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 4.5 Correlation Matrix: ESG Dimensions vs Financial Metrics

In [ ]:
corr_cols = ['ESG_Overall', 'ESG_Environmental', 'ESG_Social', 'ESG_Governance',
             'ProfitMargin', 'Revenue', 'MarketCap', 'GrowthRate']
corr_data = df_clean[corr_cols].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(10,8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (Upper Triangle)')
plt.tight_layout()
plt.show()

### 4.6 Industry Bubble Chart: ESG Score vs Profit Margin (Size = Market Cap)

In [ ]:
industry_agg = df_clean.groupby('Industry').agg({
    'ESG_Overall': 'mean',
    'ProfitMargin': 'mean',
    'MarketCap': 'mean'
}).reset_index()

fig, ax = plt.subplots(figsize=(10,6))
scatter = ax.scatter(
    industry_agg['ESG_Overall'], industry_agg['ProfitMargin'],
    s=industry_agg['MarketCap'] / 1e6,  # size in millions
    alpha=0.6, c=range(len(industry_agg)), cmap='tab10'
)
for i, row in industry_agg.iterrows():
    ax.annotate(row['Industry'], (row['ESG_Overall'], row['ProfitMargin']), fontsize=9)
ax.set_xlabel('Average ESG Overall Score')
ax.set_ylabel('Average Profit Margin (%)')
ax.set_title('Bubble size = Average Market Cap (million USD)')
plt.tight_layout()
plt.show()

### 4.7 Revenue vs Market Cap (Log-Log Scale)

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sns.scatterplot(data=df_clean, x='Revenue', y='MarketCap', hue='Industry', alpha=0.6, ax=ax)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Revenue (log scale)')
ax.set_ylabel('Market Cap (log scale)')
corr_val = df_clean[['Revenue', 'MarketCap']].corr().iloc[0,1]
ax.set_title(f'Revenue vs Market Cap (log-log) | Correlation: {corr_val:.2f}')
plt.tight_layout()
plt.show()

## 5. Summary Statistics

We display descriptive statistics for the filtered data (after cleaning).

In [ ]:
df_clean[corr_cols].describe()

## 6. Key Insights & Conclusions

1. **ESG vs Profit Margin** – The scatter plot shows a very weak positive correlation (slope ≈ 0.09). High ESG scores do not guarantee higher profit margins.
2. **ESG Category Comparison** – High ESG companies have only slightly higher average profit margins (≈11.0%) compared to Low ESG (≈8.5%), but the difference is small and overlaps within one standard deviation.
3. **Time Trends** – Over years, profit margins for all ESG categories have remained relatively stable, with High ESG performing marginally better in recent years.
4. **Correlation Matrix** – ESG_Overall has a very weak correlation with ProfitMargin (0.09), while Revenue and MarketCap are strongly correlated (0.84).
5. **Industry Differences** – Industries like Finance and Technology have high ESG scores but not necessarily the highest profit margins, suggesting that ESG performance is not a direct driver of profitability.
6. **Revenue vs Market Cap** – Strong positive correlation (0.84) on a log-log scale, which is expected.

**Limitations** – Cross-sectional data (cannot infer causality), missing control variables (firm size, leverage), ESG scores from a single provider. Future work should use panel data and regression with fixed effects.

## 7. Data Source

Kaggle – "Synthetic ESG & Financial Performance Dataset" by Shreyash Jagtap (accessed April 2026).